02_scvi_integration_umap.ipynb
Replication of: Shamsi, F., Piper, M., Ho, L.L. et al. Vascular smooth muscle-derived Trpv1+ progenitors are a source of cold-induced thermogenic adipocytes. Nat Metab 3, 485–495 (2021). https://doi.org/10.1038/s42255-021-00373-z

This is the script in python that uses the filtered data from R after QC to build an scvi model to cluster cells and use for subsequent analysis in R.

Pipeline stage: 2 of 3 (see repo README for full pipeline table) 01_qc_filtering_h5ad_export.R → THIS NOTEBOOK → 03_seurat_clustering_marker_id.R

In [ ]:
# Load packages

In [2]:
import scanpy as sc
import anndata as ad
import scvi
import torch
torch.set_float32_matmul_precision("high")

In [38]:
# Read in samples

In [3]:
passing_samples = [
    "H_BAT_nF_1",
    "H_BAT_nF_2",
    "H_BAT_nF_3",
    "H_BAT_nF_4"
]

base_dir = "/home/abhi/sc"

adata_list = []

for sample in passing_samples:
    a = sc.read_h5ad(f"{base_dir}/{sample}.h5ad")

    # make barcodes unique
    a.obs_names = [f"{sample}_{bc}" for bc in a.obs_names]

    adata_list.append(a)

In [40]:
# Merge samples into one adata object

In [4]:
adata = ad.concat(
    {sample: a for sample, a in zip(passing_samples, adata_list)},
    label="batch_indices",
    merge="same"
)

In [42]:
# Save raw counts into a seperate layer

In [5]:
adata.layers["counts"] = adata.X.copy()

In [44]:
# Switch to gene symbols instead of ensembl ids as gene identifiers

In [6]:
adata.var_names = adata.var["Symbol"].astype(str)
adata.var_names_make_unique()
adata.var = adata.var.rename(columns={"Symbol": "gene_symbol"}) # To avoid conlicts with gene identifier that would have suffixes added to avoid duplicates and gene name column.
adata.var.index.name = "Symbol"

In [7]:
adata.var


,ID,gene_symbol
Symbol,,
OR4F5,ENSG00000186092,OR4F5
OR4F29,ENSG00000284733,OR4F29
OR4F16,ENSG00000284662,OR4F16
SAMD11,ENSG00000187634,SAMD11
NOC2L,ENSG00000188976,NOC2L
...,...,...
AC233755.2,ENSG00000277856,AC233755.2
AC233755.1,ENSG00000275063,AC233755.1
AC240274.1,ENSG00000271254,AC240274.1


In [47]:
# Filter genes in <3 nuclei

In [8]:
sc.pp.filter_genes(
    adata,
    min_cells=3
)

In [9]:
adata.var


,ID,gene_symbol,n_cells
Symbol,,,
SAMD11,ENSG00000187634,SAMD11,28
NOC2L,ENSG00000188976,NOC2L,663
KLHL17,ENSG00000187961,KLHL17,112
PLEKHN1,ENSG00000187583,PLEKHN1,47
HES4,ENSG00000188290,HES4,177
...,...,...,...
AC007325.4,ENSG00000278817,AC007325.4,21
AC007325.2,ENSG00000277196,AC007325.2,8
AL354822.1,ENSG00000278384,AL354822.1,136


In [50]:
# Select top 2000 higly variable genes based on counts

In [10]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor="cell_ranger"
)
print("Number of HVGs:", adata.var["highly_variable"].sum())

adata_hvg = adata[:, adata.var["highly_variable"]].copy()

Number of HVGs: 2000


In [52]:
# Setup scVI

In [11]:
scvi.model.SCVI.setup_anndata(
    adata_hvg,
    layer="counts",
    batch_key="batch_indices"
)

In [54]:
# Model parameters

In [12]:
model = scvi.model.SCVI(
    adata_hvg,
    n_latent=20,
    gene_likelihood="nb"
)

In [56]:
# Training the model

In [13]:
model.train(
    max_epochs=400,
    accelerator="gpu",
    devices=1,
    plan_kwargs={
        "lr":1e-3
    }
)

/home/abhi/miniconda3/envs/sc/lib/python3.11/site-packages/scvi/dataloaders/_data_splitting.py:258: UserWarning: 1 cells moved from training set to validation set. if you want to avoid it please use train_size parameter during train.
  self.n_train, self.n_val = validate_data_split(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/abhi/miniconda3/envs/sc/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/abhi/miniconda3/envs/sc/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' doe

Training:   0%|          | 0/400 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=400` reached.


In [15]:
# Extract latent representations from the model


In [22]:
adata.obsm["X_scVI"] = model.get_latent_representation(adata_hvg)

In [59]:
# Compute neighbours

In [23]:
sc.pp.neighbors(
    adata,
    use_rep="X_scVI",
    n_neighbors=15
)

In [61]:
# Compute UMAP

In [24]:
sc.tl.umap(
    adata,
    min_dist=0.3
)

In [ ]:
# Save the adata object

In [25]:
adata.write("H_BAT_nF_integrated.h5ad")